# FIT4112 · Lab 1 · Notebook 02 — Tạo ma trận giỏ hàng

**Đầu vào:** `data/processed/cleaned_transactions.csv` từ Notebook 01.

**Đầu ra:** `data/processed/basket_bool.csv`.

Mỗi **hàng** là một hóa đơn; mỗi **cột** là một sản phẩm; giá trị **1** nghĩa là hóa đơn có sản phẩm đó, **0** là không.

In [ ]:
from pathlib import Path
import pandas as pd

def find_project_root():
    """Tìm thư mục gốc dù notebook chạy từ root hay notebooks/."""
    cwd = Path.cwd().resolve()
    for folder in [cwd, *cwd.parents]:
        if (folder / "data" / "raw" / "transactions_demo.csv").exists():
            return folder
    raise FileNotFoundError("Không tìm thấy data/raw/transactions_demo.csv. Hãy clone đúng repo.")

ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## 1. Đọc dữ liệu đã làm sạch

Nếu lỗi `FileNotFoundError`, hãy chạy notebook 01 trước.

In [ ]:
source_file = OUT_DIR / "cleaned_transactions.csv"
if not source_file.exists():
    raise FileNotFoundError("Hãy chạy notebook 01 để tạo cleaned_transactions.csv")
df = pd.read_csv(source_file, dtype={"InvoiceNo":"string","StockCode":"string","CustomerID":"string"})
display(df[["InvoiceNo", "Description", "Quantity"]].head(8))

## 2. Chọn kích thước xử lý

Dữ liệu thật có thể rất lớn. Giữ `MAX_INVOICES` và `TOP_PRODUCTS` để thực hành nhanh. `None` nghĩa là không giới hạn. Với dữ liệu demo, dùng toàn bộ dữ liệu.

In [ ]:
MAX_INVOICES = 2000          # dữ liệu thật: thử 1000–2000 trước
TOP_PRODUCTS = 100           # dữ liệu thật: 50–100 trước
if df["InvoiceNo"].nunique() <= 100:
    MAX_INVOICES = None
    TOP_PRODUCTS = None

work = df.copy()
if MAX_INVOICES is not None:
    keep_invoices = work["InvoiceNo"].drop_duplicates().head(MAX_INVOICES)
    work = work[work["InvoiceNo"].isin(keep_invoices)]
if TOP_PRODUCTS is not None:
    keep_products = work["Description"].value_counts().head(TOP_PRODUCTS).index
    work = work[work["Description"].isin(keep_products)]
print("Hóa đơn:", work["InvoiceNo"].nunique(), "| Sản phẩm:", work["Description"].nunique())

## 3. Chuyển sang bảng hóa đơn × sản phẩm

In [ ]:
quantities = work.groupby(["InvoiceNo","Description"])["Quantity"].sum().unstack(fill_value=0)
basket_bool = quantities.gt(0)
print("Kích thước basket:", basket_bool.shape)
display(basket_bool.head().astype(int))
print("Tỷ lệ ô có sản phẩm:", round(basket_bool.to_numpy().mean(), 4))

## 4. Lưu và kiểm tra

**Vì sao là boolean?** Apriori chỉ cần biết một mặt hàng có xuất hiện trong hóa đơn hay không, không dùng trực tiếp số lượng.

In [ ]:
assert basket_bool.shape[0] > 0 and basket_bool.shape[1] > 1
assert basket_bool.to_numpy().dtype == bool
basket_path = OUT_DIR / "basket_bool.csv"
basket_bool.astype(int).to_csv(basket_path, index=True)
print("Đã lưu:", basket_path)

## Tự kiểm tra

1. Trong basket, một dòng tương ứng với **một hóa đơn hay một khách hàng**?
2. Cùng một sản phẩm mua 3 chiếc sẽ được mã hóa thành giá trị nào?
3. Nếu chọn quá ít sản phẩm, những luật liên quan đến sản phẩm bị loại sẽ ra sao?

**Tiếp theo:** mở `03_apriori_rules.ipynb`.